## 1、自定义State，实现Model调用次数统计

In [ ]:
from langchain.agents import AgentState


# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: int  # 模型调用次数

## 2、在工具中访问state

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_model_call_count(runtime: ToolRuntime):
    """A tool that count model call """

    # 获取state中的数据
    count = runtime.state.get("model_call_count", 0)
    print("model_call_count:", count)

    # 更新State中的数据
    update= {
        "model_call_count": count + 1, # 更新count值
        "messages": [ # 工具执行结束必须返回一条ToolMessage
            ToolMessage(
                content="Successfully updated agent state",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }

    return Command(update=update)

## 3、完整代码

In [1]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import AgentState
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

# 定义自定义State结构
class CustomState(AgentState):
    """Agent的任务状态"""
    model_call_count: int  # 模型调用次数

@tool
def update_model_call_count(runtime: ToolRuntime):
    """A tool that count model call """

    # 获取state中的数据
    count = runtime.state.get("model_call_count", 0)
    print("model_call_count:", count)

    # 更新State数据
    update = {
        "model_call_count": count + 1,
        "messages": [
            ToolMessage(
                content="Successfully updated agent state",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }

    return Command(update=update)

agent = create_agent(
    "deepseek-flash",
    tools=[update_model_call_count],
    state_schema=CustomState,  # 告诉Agent使用自定义的state
    checkpointer=InMemorySaver(),
    system_prompt="你是AI助手，你需要调用工具记录模型调用次数"
)


config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi,我是虎哥")]},
    config
)

for message in response['messages']:
    message.pretty_print()


response = agent.invoke(
    {"messages": [HumanMessage(content="还记得我是谁吗")]},
    config
)

for message in response['messages']:
    message.pretty_print()

model_call_count: 0
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_pn46YoXe2kIFtLp5S7Na4514)
 Call ID: call_00_pn46YoXe2kIFtLp5S7Na4514
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Successfully updated agent state
================================== Ai Message ==================================

虎哥，你好！很高兴见到你，有什么我可以帮你的吗？
model_call_count: 1
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_pn46YoXe2kIFtLp5S7Na4514)
 Call ID: call_00_pn46YoXe2kIFtLp5S7Na4514
  Args:
================================= Tool Message =================================
Name: update_model_call_count

Succes

In [2]:
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi,我是虎哥', additional_kwargs={}, response_metadata={}, id='f47e529b-6f33-4ade-aa07-f3e564672ba0'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'The user greets me. I should respond and call the tool to record model call count.'}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 292, 'total_tokens': 335, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 292}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '9a77a9b3-001f-40a0-99df-4736a6880687', 'finish_reason': 'tool_calls', 'logp